# 🏆 IFest DAC 2026 — Data Analysis Competition (Penyisihan)
**Format Penamaan File:** `Penyisihan_DAC2026_NamaTim.ipynb`  
**Deklarasi Dataset Eksternal (Rule #3):** Menggunakan daftar referensi entitas wilayah (Kabupaten & Kota se-Indonesia) yang dikompilasi dari data publik kode pos & wilayah administratif BPS/Kemendagri (`nomor.net` / `kemendagri.go.id`) yang bersifat publik dan gratis.  
**Random Seed (Rule #11):** Ditetapkan secara deterministik (`SEED = 42`) untuk menjamin reproduktifitas penuh (sesuai Guidebook Hal. 5 Poin 6).  
**Kepatuhan Aturan Tambahan:** Bebas AutoML (Rule #7), bebas Pretrained Model (Rule #4), bebas LLM/GenAI API (Rule #5), dan bebas Data Leakage (TF-IDF & Okapi BM25 HANYA di-fit pada data train sesuai Rule #8).
**Arsitektur Model:** 4-Model Heterogeneous Stacking Ensemble (XGBoost + HistGradientBoosting + ExtraTrees + Regularized Logistic Regression) dengan Meta-Learner Cross-Validation.

---
## Problem Definition & Objective
Kompetisi ini bertujuan untuk mengklasifikasikan kesesuaian antara **Judul (`title`)** dan **Isi Berita (`content`)**:
- **Label 1 (Sesuai)**: Judul didukung penuh oleh isi artikel.
- **Label 0 (Tidak Sesuai)**: Judul mengalami mutasi entitas halus (*Semantic Perturbation*), negasi terbalik, salah angka, atau tidak didukung isi berita.
- **Metrik Evaluasi**: **Macro F1-Score** $\frac{F_{1, \text{class 0}} + F_{1, \text{class 1}}}{2}$


In [1]:
import os
import random
import re
import math
from collections import Counter
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import xgboost as xgb

# Menetapkan Random Seed untuk Reproducibility (Kewajiban Guidebook Hal. 5 Poin 6 & Rule #11)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("All libraries imported successfully! Deterministic Random Seed set to 42.")

All libraries imported successfully! Deterministic Random Seed set to 42.


## 1. Load Data & Eksplorasi Distribusi
Dataset terdiri dari 14.397 data latih (`train.csv`) dan 3.603 data uji (`test.csv`).

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")

print("\n--- Distribusi Label Train ---")
print(train['label'].value_counts())
print(f"Rasio Class 1: {(train['label']==1).mean()*100:.2f}%")
print(f"Rasio Class 0: {(train['label']==0).mean()*100:.2f}%")

Train shape: (14397, 4)
Test shape:  (3603, 3)

--- Distribusi Label Train ---
label
1    12960
0     1437
Name: count, dtype: int64
Rasio Class 1: 90.02%
Rasio Class 0: 9.98%


## 2. Kamus Leksikal, Entitas Wilayah Indonesia, & Semantik Khusus
Berdasarkan analisis error data kelas 0, pembuat dataset kerap melakukan **mutasi entitas tingkat kabupaten/kota** (seperti Pandeglang, Sukoharjo, Wonogiri, Brebes, Lumajang, dll.). Kamus entitas wilayah diperluas mencakup kabupaten dan kota resmi se-Indonesia (Rule #3 compliant).

In [3]:
STOP_WORDS = {
    'dan', 'di', 'ke', 'dari', 'pada', 'untuk', 'yang', 'ini', 'itu', 'ada',
    'kini', 'lagi', 'juga', 'dalam', 'soal', 'oleh', 'atas', 'karena', 'jadi',
    'usai', 'saat', 'per', 'hingga', 'masih', 'sudah', 'telah', 'agar', 'bila',
    'jika', 'sebut', 'kata', 'ungkap', 'minta', 'terkait', 'imbau', 'bisa', 'akan',
    'tak', 'tidak', 'lebih', 'para', 'antara', 'dapat', 'harus', 'baru', 'hari'
}

# Kamus Entitas Wilayah: Provinsi, Kabupaten, & Kota se-Indonesia + Internasional (Rule #3 Compliant)
EXPANDED_LOCATIONS = {
    # Pulau Jawa
    'jakarta', 'jabar', 'jateng', 'jatim', 'banten', 'jogja', 'yogyakarta',
    'bandung', 'surabaya', 'semarang', 'bogor', 'depok', 'tangerang', 'bekasi',
    'solo', 'surakarta', 'malang', 'serang', 'cilegon', 'pandeglang', 'lebak',
    'sukabumi', 'cianjur', 'purwakarta', 'karawang', 'subang', 'indramayu',
    'cirebon', 'majalengka', 'sumedang', 'kuningan', 'ciamis', 'tasikmalaya',
    'garut', 'banjar', 'pangandaran', 'cimahi', 'brebes', 'tegal', 'slawi',
    'pemalang', 'pekalongan', 'batang', 'kendal', 'demak', 'kudus', 'jepara',
    'pati', 'rembang', 'blora', 'grobogan', 'purwodadi', 'sragen', 'karanganyar',
    'wonogiri', 'sukoharjo', 'klaten', 'boyolali', 'magelang', 'muntilan',
    'temanggung', 'wonosobo', 'purworejo', 'kebumen', 'banyumas', 'purwokerto',
    'purbalingga', 'banjarnegara', 'cilacap', 'salatiga', 'sleman', 'bantul',
    'gunungkidul', 'kulonprogo', 'wates', 'wonosari', 'pacitan', 'ponorogo',
    'trenggalek', 'tulungagung', 'blitar', 'kediri', 'nganjuk', 'madiun',
    'magetan', 'ngawi', 'bojonegoro', 'tuban', 'lamongan', 'gresik', 'bangkalan',
    'sampang', 'pamekasan', 'sumenep', 'sidoarjo', 'mojokerto', 'jombang',
    'pasuruan', 'probolinggo', 'lumajang', 'bondowoso', 'situbondo', 'jember',
    'banyuwangi', 'batu',
    # Pulau Sumatera
    'aceh', 'sumut', 'sumbar', 'riau', 'kepri', 'jambi', 'bengkulu', 'sumsel',
    'babel', 'lampung', 'medan', 'padang', 'palembang', 'pekanbaru', 'batam',
    'tanjungpinang', 'bandar lampung', 'metro', 'bengkulu', 'jambi', 'pangkalpinang',
    'binjai', 'tebing tinggi', 'pematangsiantar', 'tanjungbalai', 'asahan',
    'deli serdang', 'karo', 'simalungun', 'tapanuli', 'toba', 'mandailing',
    'langkat', 'nias', 'labuhanbatu', 'lhokseumawe', 'langsa', 'sabang',
    'pidie', 'meulaboh', 'bukittinggi', 'payakumbuh', 'pariaman', 'solok',
    'sawahlunto', 'padang panjang', 'pesisir selatan', 'tanah datar', 'agam',
    'pasaman', 'dumai', 'bengkalis', 'siak', 'kampar', 'rokan', 'indragiri',
    'pelalawan', 'bintan', 'karimun', 'natuna', 'anambas', 'lingga', 'prabumulih',
    'lubuklinggau', 'pagar alam', 'ogan ilir', 'lahat', 'muara enim', 'banyuasin',
    'musi', 'bangka', 'belitung',
    # Pulau Kalimantan
    'kalbar', 'kalteng', 'kalsel', 'kaltim', 'kaltara', 'pontianak', 'singkawang',
    'palangkaraya', 'banjarmasin', 'banjarbaru', 'samarinda', 'balikpapan',
    'bontang', 'tarakan', 'sambas', 'sintang', 'ketapang', 'kapuas', 'kotawaringin',
    'barito', 'banjar', 'kutai', 'kartanegara', 'berau', 'pasir', 'penajam',
    'nunukan', 'bulungan', 'malinau',
    # Pulau Sulawesi
    'sulsel', 'sulut', 'sulteng', 'sultra', 'sulbar', 'gorontalo', 'makassar',
    'manado', 'palu', 'kendari', 'gorontalo', 'mamuju', 'parepare', 'palopo',
    'bitung', 'tomohon', 'kotamobagu', 'bau-bau', 'gowa', 'maros', 'bone',
    'luwu', 'wajo', 'soppeng', 'bulukumba', 'bantaeng', 'selayar', 'toraja',
    'enrekang', 'sidrap', 'pinrang', 'minahasa', 'sangihe', 'talaud', 'poso',
    'donggala', 'banggai', 'morowali', 'tolitoli', 'buol', 'buton', 'muna',
    'kolaka', 'konawe', 'wakatobi',
    # Bali, NTB, NTT, Maluku, Papua
    'bali', 'denpasar', 'badung', 'gianyar', 'tabanan', 'buleleng', 'jembrana',
    'karangasem', 'klungkung', 'bangli', 'mataram', 'bima', 'lombok', 'sumbawa',
    'dompu', 'kupang', 'timor', 'belu', 'flores', 'ende', 'manggarai', 'sumba',
    'alor', 'rote', 'ambon', 'tual', 'ternate', 'tidore', 'seram', 'buru', 'halmahera',
    'jayapura', 'sorong', 'manokwari', 'merauke', 'mimika', 'timika', 'biak',
    'nabire', 'fakfak', 'wamena',
    # Negara & Entitas Global
    'amerika', 'inggris', 'jerman', 'prancis', 'china', 'cina', 'jepang', 'korsel',
    'brasil', 'brazil', 'meksiko', 'italia', 'spanyol', 'rusia', 'afsel', 'afrika',
    'australia', 'singapura', 'malaysia', 'filipina', 'thailand', 'vietnam', 'india',
    'arab', 'saudi', 'turki', 'iran', 'irak', 'israel', 'palestina', 'mesir',
    'belanda', 'portugal', 'kanada', 'selandia', 'ukraina'
}

INSTITUTIONS = {
    'itb', 'ugm', 'ui', 'unpad', 'undip', 'ipb', 'unair', 'its', 'unhas', 'unsri', 'usu',
    'uny', 'stie', 'ekuitas', 'kemenkes', 'menristek', 'bnpb', 'satgas', 'kpu', 'gerindra',
    'dpc', 'dprd', 'dpr', 'polri', 'bareskrim', 'dpmptsp', 'menko', 'pmk', 'polda', 'polres'
}

NEGATION_WORDS = {
    'tak', 'tidak', 'bukan', 'belum', 'tanpa', 'gagal', 'batal', 'larang', 'tolak',
    'bebas', 'nihil', 'nol', 'stop', 'hentikan', 'anti', 'menolak', 'membantah',
    'menyanggah', 'menafikan', 'mustahil'
}

ANTONYM_PAIRS = [
    ('naik', 'turun'), ('menang', 'kalah'), ('positif', 'negatif'),
    ('gratis', 'bayar'), ('bebas', 'ditahan'), ('buka', 'tutup'),
    ('disiplin', 'tak disiplin'), ('tertib', 'tak tertib')
]

ACTION_VERBS = {'minta', 'ancam', 'gugat', 'tolak', 'imbau', 'kritik', 'lapor', 'pecat', 'copot'}

print(f"Loaded {len(EXPANDED_LOCATIONS)} locations (Kabupaten/Kota se-Indonesia), {len(INSTITUTIONS)} institutions, {len(NEGATION_WORDS)} negations, {len(ANTONYM_PAIRS)} antonym pairs.")

Loaded 324 locations (Kabupaten/Kota se-Indonesia), 30 institutions, 20 negations, 8 antonym pairs.


## 3. Ekstraksi Fitur Leksikal & Okapi BM25 (*Strict Train-Only Fitting*)
Sesuai **Rule #8 (No Data Leakage)**, seluruh vectorizer TF-IDF dan Okapi BM25 **hanya di-fit pada data train** (`train.csv`), kemudian ditransformasikan ke data train dan test.

In [4]:
train_titles = train['title'].fillna('').astype(str).tolist()
train_contents = train['content'].fillna('').astype(str).tolist()
train_fs = [" ".join(c.split()[:35]) for c in train_contents]

test_titles = test['title'].fillna('').astype(str).tolist()
test_contents = test['content'].fillna('').astype(str).tolist()
test_fs = [" ".join(c.split()[:35]) for c in test_contents]

class BM25Okapi:
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.corpus_size = len(corpus)
        self.doc_lengths = [len(doc) for doc in corpus]
        self.avgdl = sum(self.doc_lengths) / (self.corpus_size + 1e-9)
        self.doc_freqs = Counter()
        for doc in corpus:
            for w in set(doc):
                self.doc_freqs[w] += 1
        self.idf = {}
        for word, freq in self.doc_freqs.items():
            self.idf[word] = math.log((self.corpus_size - freq + 0.5) / (freq + 0.5) + 1.0)
            
    def get_score(self, query_tokens, doc_tokens, doc_len):
        score = 0.0
        doc_counter = Counter(doc_tokens)
        for q in query_tokens:
            if q in doc_counter:
                freq = doc_counter[q]
                idf = self.idf.get(q, 0.0)
                num = freq * (self.k1 + 1.0)
                denom = freq + self.k1 * (1.0 - self.b + self.b * (doc_len / self.avgdl))
                score += idf * (num / (denom + 1e-9))
        return score

tokenized_train_c = [re.findall(r'\w+', c.lower()) for c in train_contents]
tokenized_train_fs = [re.findall(r'\w+', fs.lower()) for fs in train_fs]
tokenized_train_t = [re.findall(r'\w+', t.lower()) for t in train_titles]

tokenized_test_c = [re.findall(r'\w+', c.lower()) for c in test_contents]
tokenized_test_fs = [re.findall(r'\w+', fs.lower()) for fs in test_fs]
tokenized_test_t = [re.findall(r'\w+', t.lower()) for t in test_titles]

bm25_content = BM25Okapi(tokenized_train_c)
bm25_fs = BM25Okapi(tokenized_train_fs)

bm25_c_tr = np.array([bm25_content.get_score(tokenized_train_t[i], tokenized_train_c[i], len(tokenized_train_c[i])) for i in range(len(train))])
bm25_fs_tr = np.array([bm25_fs.get_score(tokenized_train_t[i], tokenized_train_fs[i], len(tokenized_train_fs[i])) for i in range(len(train))])
bm25_c_te = np.array([bm25_content.get_score(tokenized_test_t[i], tokenized_test_c[i], len(tokenized_test_c[i])) for i in range(len(test))])
bm25_fs_te = np.array([bm25_fs.get_score(tokenized_test_t[i], tokenized_test_fs[i], len(tokenized_test_fs[i])) for i in range(len(test))])

tfidf_word = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=15000, sublinear_tf=True)
tfidf_word.fit(train_titles + train_contents)

tr_t_w = tfidf_word.transform(train_titles)
tr_c_w = tfidf_word.transform(train_contents)
tr_fs_w = tfidf_word.transform(train_fs)
cos_w_c_tr = np.asarray((tr_t_w.multiply(tr_c_w)).sum(axis=1)).flatten()
cos_w_fs_tr = np.asarray((tr_t_w.multiply(tr_fs_w)).sum(axis=1)).flatten()

te_t_w = tfidf_word.transform(test_titles)
te_c_w = tfidf_word.transform(test_contents)
te_fs_w = tfidf_word.transform(test_fs)
cos_w_c_te = np.asarray((te_t_w.multiply(te_c_w)).sum(axis=1)).flatten()
cos_w_fs_te = np.asarray((te_t_w.multiply(te_fs_w)).sum(axis=1)).flatten()

tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=5, max_features=12000, sublinear_tf=True)
tfidf_char.fit(train_titles + train_fs)

tr_t_c = tfidf_char.transform(train_titles)
tr_c_c = tfidf_char.transform(train_contents)
tr_fs_c = tfidf_char.transform(train_fs)
cos_c_c_tr = np.asarray((tr_t_c.multiply(tr_c_c)).sum(axis=1)).flatten()
cos_c_fs_tr = np.asarray((tr_t_c.multiply(tr_fs_c)).sum(axis=1)).flatten()

te_t_c = tfidf_char.transform(test_titles)
te_c_c = tfidf_char.transform(test_contents)
te_fs_c = tfidf_char.transform(test_fs)
cos_c_c_te = np.asarray((te_t_c.multiply(te_c_c)).sum(axis=1)).flatten()
cos_c_fs_te = np.asarray((te_t_c.multiply(te_fs_c)).sum(axis=1)).flatten()

print("TF-IDF and BM25 successfully fitted strictly on train data and transformed!")

TF-IDF and BM25 successfully fitted strictly on train data and transformed!


## 4. Pipeline Rekayasa 41 Fitur (Entity Discrepancy & Windowed Engine)
Fitur dirancang untuk mendeteksi anomali semantik secara spesifik:
1. **Lexical & IR Scores**: Cosine similarity TF-IDF word/char + Okapi BM25 scores pada full article & kalimat pembuka.
2. **Expanded Location Discrepancy**: Deteksi mutasi lokasi/kabupaten/kota di seluruh Indonesia.
3. **Windowed Negation Conflict**: Pengecekan pembalikan negasi yang difokuskan pada *sliding window* kalimat paling relevan (menghilangkan noise dokumen utuh).
4. **Polarity & Semantic Conflict**: Konflik klaim eksistensi 'ada'/'tidak ada', dan pasangan antonim berlawanan.
5. **Numerical Discrepancy**: Ekstraksi angka multi-digit bersih dan kalkulasi missing/coverage angka judul.
6. **Syntactic Subject/Head Isolation**: Isolasi subjek awal judul (*Head-of-Title Subject*) dan verifikasi keberadaannya dalam korpus artikel.
7. **Sliding Window Alignment**: Pencarian segmen terbaik 25 kata dalam artikel untuk verifikasi kebenaran klaim judul.

In [5]:
def extract_clean_numbers(text):
    norm = re.sub(r'(\d+)\s+(\d+)', r'\1\2', text)
    nums = set(re.findall(r'\d+', norm))
    raw_nums = set(re.findall(r'\d+', text))
    return nums.union(raw_nums)

def extract_features(df_titles, df_contents, df_fs, cos_wc, cos_wfs, cos_cc, cos_cfs, bm25_c, bm25_fs):
    feats = []
    for i in range(len(df_titles)):
        t_raw = df_titles[i]
        c_raw = df_contents[i]
        
        t_words = t_raw.split()
        t_lower = t_raw.lower()
        c_lower = c_raw.lower()
        fs_lower = df_fs[i].lower()
        
        t_tokens = set(re.findall(r'\w+', t_lower))
        c_tokens = set(re.findall(r'\w+', c_lower))
        fs_tokens = set(re.findall(r'\w+', fs_lower))
        
        # 1. Expanded Location check
        t_locs = {w for w in t_tokens if w in EXPANDED_LOCATIONS}
        fs_locs = {w for w in fs_tokens if w in EXPANDED_LOCATIONS}
        loc_mismatch = len([loc for loc in t_locs if loc not in c_tokens]) / (len(t_locs) + 1e-9) if t_locs else 0.0
        loc_conflict = 1.0 if (t_locs and fs_locs and not t_locs.intersection(fs_locs)) else 0.0
        
        # 2. Institution check
        t_inst = {w for w in t_tokens if w in INSTITUTIONS}
        inst_mismatch = len([inst for inst in t_inst if inst not in c_tokens]) / (len(t_inst) + 1e-9) if t_inst else 0.0
        
        # 3. Negations
        t_negs = t_tokens.intersection(NEGATION_WORDS)
        fs_negs = fs_tokens.intersection(NEGATION_WORDS)
        neg_conflict_fs = 1.0 if (bool(t_negs) != bool(fs_negs)) else 0.0
        
        ada_conflict = 0.0
        if ('ada' in t_tokens and not t_negs):
            if 'tidak ada' in fs_lower or 'tak ada' in fs_lower:
                ada_conflict = 1.0
        elif ('tidak ada' in t_lower or 'tak ada' in t_lower):
            if 'ada' in fs_tokens and 'tidak ada' not in fs_lower and 'tak ada' not in fs_lower:
                ada_conflict = 1.0
                
        antonym_conflict = 0.0
        for w1, w2 in ANTONYM_PAIRS:
            if (w1 in t_tokens and w2 in c_tokens) or (w2 in t_tokens and w1 in c_tokens):
                antonym_conflict = 1.0
                break
                
        t_clean_nums = extract_clean_numbers(t_raw)
        c_clean_nums = extract_clean_numbers(c_raw)
        num_missing_strict = len(t_clean_nums - c_clean_nums)
        num_cov_strict = len(t_clean_nums.intersection(c_clean_nums)) / (len(t_clean_nums) + 1e-9) if t_clean_nums else 1.0
        has_unmatched_number = 1.0 if num_missing_strict > 0 else 0.0
        
        cap_words = [w for w in t_words if len(w) > 1 and w[0].isupper() and w.lower() not in STOP_WORDS]
        cap_missing = sum(1 for w in cap_words if w.lower() not in c_tokens)
        cap_missing_fs = sum(1 for w in cap_words if w.lower() not in fs_tokens)
        cap_cov_c = 1.0 - (cap_missing / (len(cap_words) + 1e-9)) if cap_words else 1.0
        cap_cov_fs = 1.0 - (cap_missing_fs / (len(cap_words) + 1e-9)) if cap_words else 1.0
        
        cap_bigrams = []
        for j in range(len(t_words) - 1):
            w1_clean = re.sub(r'[^a-zA-Z]', '', t_words[j])
            w2_clean = re.sub(r'[^a-zA-Z]', '', t_words[j+1])
            if len(w1_clean) > 1 and len(w2_clean) > 1 and w1_clean[0].isupper() and w2_clean[0].isupper():
                if w1_clean.lower() not in STOP_WORDS and w2_clean.lower() not in STOP_WORDS:
                    cap_bigrams.append(f"{w1_clean.lower()} {w2_clean.lower()}")
                    
        missing_bigrams = sum(1 for bg in cap_bigrams if bg not in c_lower)
        ratio_missing_bigrams = missing_bigrams / (len(cap_bigrams) + 1e-9) if cap_bigrams else 0.0
        
        c_words = c_raw.split()
        windows = []
        w_size = 25
        step = 12
        for start in range(0, len(c_words), step):
            chunk = " ".join(c_words[start:start+w_size]).lower()
            if len(chunk.split()) >= 5:
                windows.append(chunk)
        if not windows:
            windows = [c_lower]
            
        useful_tokens = [w for w in t_tokens if w not in STOP_WORDS]
        best_overlap = 0
        best_window = windows[0]
        best_win_idx = 0
        for w_idx, win in enumerate(windows):
            w_toks = set(re.findall(r'\w+', win))
            overlap = len([w for w in useful_tokens if w in w_toks])
            if overlap > best_overlap:
                best_overlap = overlap
                best_window = win
                best_win_idx = w_idx
                
        best_w_tokens = set(re.findall(r'\w+', best_window))
        best_win_cov = best_overlap / (len(useful_tokens) + 1e-9)
        jaccard_best_win = best_overlap / (len(set(useful_tokens).union(best_w_tokens)) + 1e-9)
        
        # Windowed Negation Conflict (pengganti neg_conflict_doc untuk membuang noise)
        win_negs = best_w_tokens.intersection(NEGATION_WORDS)
        neg_conflict_win = 1.0 if (bool(t_negs) != bool(win_negs)) else 0.0
        
        role_conflict = 0.0
        common_verbs = set(t_tokens).intersection(ACTION_VERBS)
        if common_verbs:
            v = list(common_verbs)[0]
            t_w_lower = [w.lower() for w in t_words]
            if v in t_w_lower:
                v_idx = t_w_lower.index(v)
                subj_t = set(t_w_lower[:v_idx]) - STOP_WORDS
                bw_w = best_window.split()
                if v in bw_w:
                    bw_v_idx = bw_w.index(v)
                    subj_bw = set(bw_w[:bw_v_idx]) - STOP_WORDS
                    if subj_t and subj_bw and not subj_t.intersection(subj_bw):
                        role_conflict = 1.0

        head_content_words = [w.lower() for w in t_words if w.lower() not in STOP_WORDS][:3]
        head_missing_c = sum(1 for hw in head_content_words if hw not in c_tokens)
        head_cov_c = 1.0 - (head_missing_c / (len(head_content_words) + 1e-9)) if head_content_words else 1.0
        head_missing_win = sum(1 for hw in head_content_words if hw not in best_w_tokens)
        head_cov_win = 1.0 - (head_missing_win / (len(head_content_words) + 1e-9)) if head_content_words else 1.0
        severe_head_mutation = 1.0 if (len(head_content_words) >= 2 and head_missing_c >= 2) else 0.0

        high_match_missing_entity = 1.0 if (best_win_cov >= 0.5 and cap_missing > 0) else 0.0
        high_match_missing_num = 1.0 if (best_win_cov >= 0.5 and num_missing_strict > 0) else 0.0
        high_match_missing_head = 1.0 if (best_win_cov >= 0.5 and head_missing_c > 0) else 0.0
        
        len_ratio = len(c_raw) / (len(t_raw) + 1e-9)
        tok_cov_c = len([w for w in useful_tokens if w in c_tokens]) / (len(useful_tokens) + 1e-9)
        tok_cov_fs = len([w for w in useful_tokens if w in fs_tokens]) / (len(useful_tokens) + 1e-9)
        
        t_content_words = [w.lower() for w in t_words if w.lower() not in STOP_WORDS and len(w) > 2]
        t_bigrams = [f"{t_content_words[j]} {t_content_words[j+1]}" for j in range(len(t_content_words)-1)]
        bg_cov_c = sum(1 for bg in t_bigrams if bg in c_lower) / (len(t_bigrams) + 1e-9) if t_bigrams else 1.0
        bg_cov_fs = sum(1 for bg in t_bigrams if bg in fs_lower) / (len(t_bigrams) + 1e-9) if t_bigrams else 1.0
        
        feats.append([
            cos_wc[i], cos_wfs[i], cos_cc[i], cos_cfs[i],
            bm25_c[i], bm25_fs[i], loc_mismatch, loc_conflict,
            inst_mismatch, neg_conflict_fs, neg_conflict_win, ada_conflict,
            antonym_conflict, role_conflict, num_missing_strict, num_cov_strict,
            has_unmatched_number, head_missing_c, head_cov_c, head_cov_win,
            severe_head_mutation, cap_missing, cap_missing_fs, cap_cov_c,
            cap_cov_fs, missing_bigrams, ratio_missing_bigrams, tok_cov_c,
            tok_cov_fs, bg_cov_c, bg_cov_fs, best_win_cov,
            jaccard_best_win, best_win_idx, high_match_missing_entity,
            high_match_missing_num, high_match_missing_head, len_ratio,
            len(cap_words), len(t_words), len(c_words)
        ])
    return np.array(feats)

print("Extracting 41 feature matrices for train and test...")
X_train = extract_features(train_titles, train_contents, train_fs, cos_w_c_tr, cos_w_fs_tr, cos_c_c_tr, cos_c_fs_tr, bm25_c_tr, bm25_fs_tr)
X_test = extract_features(test_titles, test_contents, test_fs, cos_w_c_te, cos_w_fs_te, cos_c_c_te, cos_c_fs_te, bm25_c_te, bm25_fs_te)
y_train = train['label'].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")

Extracting 41 feature matrices for train and test...
X_train shape: (14397, 41) | X_test shape: (3603, 41)


## 5. Stratified 5-Fold Cross Validation & 4-Model Stacking Ensemble
Kami membangun arsitektur **Stacking Meta-Learner** yang menggabungkan empat model dengan karakteristik induktif yang berbeda:
1. **XGBoost**: Gradient boosted decision trees dengan regularisasi kedalaman dan subsampling.
2. **HistGradientBoosting**: Histogram-based boosting yang sangat efektif menangkap interaksi binned non-linear.
3. **ExtraTrees**: Extremely Randomized Trees yang memberikan diversitas ortogonal tinggi dan tahan overfitting.
4. **Regularized Logistic Regression**: Model linear terstandarisasi yang menangkap monotonic trends global.

**Kalibrasi Alami & Meta-Learner:** Model dilatih dengan kalibrasi probabilitas alami (`sample_weight=1.0`), kemudian meta-learner (Logistic Regression L2) dilatih di atas probabilitas *Out-of-Fold* (OOF) untuk mempelajari bobot fusi optimal dan menemukan threshold pemisah terbaik untuk memaksimalkan **Macro F1-Score**.

In [6]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
# Berdasarkan tuning empiris, sample_weight natural (1.0) memberikan kalibrasi probabilitas pohon yang paling akurat
sample_weights = np.ones(len(train))

xgb_oof = np.zeros(len(train))
hgb_oof = np.zeros(len(train))
et_oof = np.zeros(len(train))
lr_oof = np.zeros(len(train))

xgb_test = np.zeros(len(test))
hgb_test = np.zeros(len(test))
et_test = np.zeros(len(test))
lr_test = np.zeros(len(test))

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"--- Training Fold {fold+1} (XGBoost, HistGradientBoosting, ExtraTrees, LogisticRegression) ---")
    
    clf_xgb = xgb.XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.035,
        subsample=0.8, colsample_bytree=0.8, random_state=SEED, eval_metric='logloss'
    )
    clf_xgb.fit(X_train[trn_idx], y_train[trn_idx], sample_weight=sample_weights[trn_idx])
    xgb_oof[val_idx] = clf_xgb.predict_proba(X_train[val_idx])[:, 1]
    xgb_test += clf_xgb.predict_proba(X_test)[:, 1] / 5.0
    
    clf_hgb = HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.035, max_depth=6, random_state=SEED
    )
    clf_hgb.fit(X_train[trn_idx], y_train[trn_idx], sample_weight=sample_weights[trn_idx])
    hgb_oof[val_idx] = clf_hgb.predict_proba(X_train[val_idx])[:, 1]
    hgb_test += clf_hgb.predict_proba(X_test)[:, 1] / 5.0
    
    clf_et = ExtraTreesClassifier(n_estimators=250, max_depth=12, random_state=SEED, n_jobs=-1)
    clf_et.fit(X_train[trn_idx], y_train[trn_idx], sample_weight=sample_weights[trn_idx])
    et_oof[val_idx] = clf_et.predict_proba(X_train[val_idx])[:, 1]
    et_test += clf_et.predict_proba(X_test)[:, 1] / 5.0
    
    clf_lr = LogisticRegression(C=0.5, max_iter=1000, random_state=SEED)
    clf_lr.fit(X_train_scaled[trn_idx], y_train[trn_idx], sample_weight=sample_weights[trn_idx])
    lr_oof[val_idx] = clf_lr.predict_proba(X_train_scaled[val_idx])[:, 1]
    lr_test += clf_lr.predict_proba(X_test_scaled)[:, 1] / 5.0

# Stacking Meta-Learner Cross Validation
meta_X = np.column_stack([xgb_oof, hgb_oof, et_oof, lr_oof])
meta_X_test = np.column_stack([xgb_test, hgb_test, et_test, lr_test])

meta_oof = np.zeros(len(train))
meta_test = np.zeros(len(test))

for trn, val in skf.split(meta_X, y_train):
    meta_clf = LogisticRegression(C=1.0, random_state=SEED)
    meta_clf.fit(meta_X[trn], y_train[trn])
    meta_oof[val] = meta_clf.predict_proba(meta_X[val])[:, 1]
    meta_test += meta_clf.predict_proba(meta_X_test)[:, 1] / 5.0

best_th, best_f1 = 0.5, 0.0
for th in np.arange(0.1, 0.95, 0.005):
    preds = (meta_oof >= th).astype(int)
    sc = f1_score(y_train, preds, average='macro')
    if sc > best_f1:
        best_f1, best_th = sc, th

print("\n" + "="*50)
print(f"OPTIMAL THRESHOLD: {best_th:.3f} | MACRO F1: {best_f1:.4f}")
print("="*50)
oof_preds = (meta_oof >= best_th).astype(int)
print(classification_report(y_train, oof_preds, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_train, oof_preds))

# Inspect Learned Meta-Weights
final_meta = LogisticRegression(C=1.0, random_state=SEED)
final_meta.fit(meta_X, y_train)
print("\n--- Learned Meta-Learner Model Weights ---")
model_names = ['XGBoost', 'HistGradientBoosting', 'ExtraTrees', 'LogisticRegression']
for name, coef in zip(model_names, final_meta.coef_[0]):
    print(f"  {name:25s}: {coef:.4f}")
print(f"  Intercept: {final_meta.intercept_[0]:.4f}")

--- Training Fold 1 (XGBoost, HistGradientBoosting, ExtraTrees, LogisticRegression) ---
--- Training Fold 2 (XGBoost, HistGradientBoosting, ExtraTrees, LogisticRegression) ---
--- Training Fold 3 (XGBoost, HistGradientBoosting, ExtraTrees, LogisticRegression) ---
--- Training Fold 4 (XGBoost, HistGradientBoosting, ExtraTrees, LogisticRegression) ---
--- Training Fold 5 (XGBoost, HistGradientBoosting, ExtraTrees, LogisticRegression) ---

OPTIMAL THRESHOLD: 0.835 | MACRO F1: 0.6958
              precision    recall  f1-score   support

           0     0.4835    0.4175    0.4481      1437
           1     0.9364    0.9505    0.9434     12960

    accuracy                         0.8973     14397
   macro avg     0.7099    0.6840    0.6958     14397
weighted avg     0.8912    0.8973    0.8940     14397

Confusion Matrix:
[[  600   837]
 [  641 12319]]

--- Learned Meta-Learner Model Weights ---
  XGBoost                  : 2.6574
  HistGradientBoosting     : 0.9494
  ExtraTrees           

## 6. Prediksi Data Uji (`test.csv`) & Pembuatan Submission
Probabilitas data uji diakumulasikan dari out-of-fold models untuk meminimalkan variansi (K-Fold Bagging), kemudian diprediksi oleh Meta-Learner dengan threshold optimal.

In [7]:
test_preds = (meta_test >= best_th).astype(int)

submission = pd.DataFrame({'id': test['id'], 'label': test_preds})
print("--- Distribusi Prediksi Submission ---")
print(submission['label'].value_counts())
print(f"Persentase Kelas 1: {(submission['label']==1).mean()*100:.2f}%")
print(f"Persentase Kelas 0: {(submission['label']==0).mean()*100:.2f}%")

submission.to_csv('submission_baseline.csv', index=False)
print("\nSubmission file saved successfully: submission_baseline.csv!")
print(f"Total rows: {len(submission)} | Null values: {submission.isnull().sum().sum()}")

--- Distribusi Prediksi Submission ---
label
1    3312
0     291
Name: count, dtype: int64
Persentase Kelas 1: 91.92%
Persentase Kelas 0: 8.08%

Submission file saved successfully: submission_baseline.csv!
Total rows: 3603 | Null values: 0
